# E-Commerce Data Cleaning
### Portfolio Project -- Python · Pandas · NumPy · Matplotlib

**Goal:** Take a messy, real-world-style e-commerce dataset and produce a clean,
analysis-ready version while documenting every decision.

**Problems fixed:**
- Duplicate rows
- Missing values (NaN)
- Inconsistent text formatting (categories, cities)
- Prices stored as strings (`"$19.99"`, `"19,99 USD"`)
- Dates in five different formats
- Extra whitespace
- Outliers in price, quantity, and rating


## 0. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os
import random
from datetime import datetime, timedelta

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
print('Libraries ready.')

## 1. Generate the Dirty Dataset

We programmatically create **200 rows** of e-commerce product data and
inject every data-quality problem we intend to fix.

In [ ]:
random.seed(42)
np.random.seed(42)

CATEGORIES = [
    'Electronics', 'electronics', 'ELECTRONICS',
    'Clothing',    'clothing',    'CLOTHING',
    'Home & Garden', 'home & garden',
    'Sports',      'sports',      'SPORTS',
    'Books',       'books',
    'Toys',        'toys',        'TOYS',
]

CITIES = [
    'New York',   'new york',   'NEW YORK',  'NY',  'New York ',
    'Los Angeles','los angeles','LOS ANGELES','LA', ' Los Angeles',
    'Chicago',    'chicago',    'CHICAGO',   'Chi',
    'Houston',    'houston',    'HOUSTON',
    'Phoenix',    'phoenix',    'PHOENIX',
]

PRODUCTS = [
    '  Wireless Headphones  ', 'Laptop Stand', 'USB-C Hub ',
    ' Running Shoes', 'Yoga Mat  ', 'Coffee Maker',
    '  Notebook', 'Bluetooth Speaker', 'Phone Case ',
    ' Desk Lamp', 'Water Bottle', '  Backpack',
    'Smart Watch', 'Keyboard ', ' Mouse Pad', 'Monitor Stand',
    'T-Shirt  ', '  Jeans', 'Sneakers ', ' Jacket',
    'Garden Hose  ', '  Plant Pot', 'Shovel ', ' Rake',
    'Football', '  Basketball', 'Tennis Racket ', ' Swimming Goggles',
    'Python Book  ', '  Data Science Book', 'Novel ', ' Cookbook',
    'LEGO Set  ', '  Action Figure', 'Board Game ', ' Puzzle',
    'Wireless Mouse', 'Standing Desk', 'Resistance Bands ', ' Yoga Block',
]

def random_date():
    base = datetime(2023, 1, 1) + timedelta(days=random.randint(0, 730))
    fmt = random.choice(['%m/%d/%Y', '%Y-%m-%d', '%d-%m-%Y',
                          '%B %d, %Y', '%d/%m/%Y'])
    return base.strftime(fmt)

def random_price():
    v = round(random.uniform(5.0, 350.0), 2)
    fmt = random.choice(['plain', 'dollar', 'usd', 'comma', 'plain', 'plain'])
    if fmt == 'dollar': return f'${v}'
    if fmt == 'usd':    return f'{v} USD'
    if fmt == 'comma':  return str(v).replace('.', ',')
    return v

N = 180
rows = []
for i in range(N):
    rows.append({
        'order_id':       f'ORD-{str(i+1).zfill(4)}',
        'product_name':   random.choice(PRODUCTS),
        'category':       random.choice(CATEGORIES),
        'price':          random_price(),
        'quantity':       random.randint(1, 50),
        'city':           random.choice(CITIES),
        'date_added':     random_date(),
        'rating':         round(random.uniform(1.0, 5.0), 1),
        'customer_email': f'customer{i+1}@example.com',
    })

df_dirty = pd.DataFrame(rows)

# Inject outliers
df_dirty.loc[5,  'price']    = 9999.99
df_dirty.loc[12, 'price']    = '$8500'
df_dirty.loc[30, 'rating']   = 15.0
df_dirty.loc[55, 'quantity'] = 9999

# Inject missing values
missing_plan = {'product_name':8,'category':10,'price':12,
                'quantity':8,'city':10,'date_added':7,'rating':9}
for col, n in missing_plan.items():
    df_dirty.loc[random.sample(range(N), n), col] = np.nan

# Inject duplicates (20 rows)
dupes = df_dirty.sample(20, random_state=42)
df_dirty = pd.concat([df_dirty, dupes], ignore_index=True)
df_dirty = df_dirty.sample(frac=1, random_state=42).reset_index(drop=True)

os.makedirs('data', exist_ok=True)
df_dirty.to_csv('data/dirty_ecommerce.csv', index=False)
print(f'Dirty dataset created: {df_dirty.shape[0]} rows x {df_dirty.shape[1]} columns')

## 2. Inspect the Dirty Data

Before touching anything, we document exactly what is broken.

In [ ]:
df = pd.read_csv('data/dirty_ecommerce.csv', dtype=str)
print(f'Shape: {df.shape}')
df.head(8)

In [ ]:
# Summary table BEFORE cleaning
before_summary = pd.DataFrame({
    'dtype':    df.dtypes.astype(str),
    'nulls':    df.isnull().sum(),
    'null_%':   (df.isnull().mean() * 100).round(1),
    'unique':   df.nunique(),
})
print(f'Total rows     : {len(df)}')
print(f'Total nulls    : {df.isnull().sum().sum()}')
print(f'Duplicate rows : {df.duplicated().sum()}')
print()
before_summary

In [ ]:
# Sample of the messy values we need to fix
print('--- Sample price values ---')
print(df['price'].dropna().sample(10, random_state=1).tolist())
print()
print('--- Sample date_added values ---')
print(df['date_added'].dropna().sample(10, random_state=1).tolist())
print()
print('--- Unique city values (first 15) ---')
print(sorted(df['city'].dropna().unique())[:15])

## 3. Cleaning Pipeline

### Step 1 -- Remove Duplicate Rows
Exact duplicate rows add no information and inflate every count and average.

In [ ]:
n_before = len(df)
df = df.drop_duplicates()
print(f'Removed {n_before - len(df)} duplicate rows  ->  {len(df)} rows remaining')

### Step 2 -- Strip Extra Whitespace
Leading/trailing spaces cause silent mismatches: `' Electronics '` ≠ `'Electronics'`.

In [ ]:
text_cols = ['product_name', 'category', 'city', 'customer_email']
for col in text_cols:
    df[col] = df[col].str.strip()
    df[col] = df[col].replace('nan', np.nan)   # fix str(NaN) artefact

# Spot-check
print('product_name sample after strip:')
print(df['product_name'].dropna().sample(5, random_state=1).tolist())

### Step 3 -- Standardise Categories
`'electronics'`, `'ELECTRONICS'`, `'Electronics'` -> `'Electronics'`

In [ ]:
df['category'] = df['category'].str.title()
print('Unique categories:', sorted(df['category'].dropna().unique()))

### Step 4 -- Standardise City Names
Map abbreviations (`NY`, `LA`, `Chi`) and fix capitalisation.

In [ ]:
city_map = {
    'ny': 'New York', 'new york': 'New York',
    'la': 'Los Angeles', 'los angeles': 'Los Angeles',
    'chi': 'Chicago', 'chicago': 'Chicago',
    'houston': 'Houston', 'phoenix': 'Phoenix',
}

def fix_city(val):
    if pd.isna(val): return np.nan
    return city_map.get(str(val).strip().lower(), str(val).strip().title())

df['city'] = df['city'].apply(fix_city)
print('Unique cities:', sorted(df['city'].dropna().unique()))

### Step 5 -- Clean & Convert Price to Float
Remove `$`, strip `USD`, fix comma-as-decimal-separator, then cast to `float64`.

In [ ]:
def clean_price(val):
    if pd.isna(val) or str(val).strip().lower() == 'nan':
        return np.nan
    s = str(val).strip().replace('$', '').replace('USD', '').strip()
    s = s.replace(',', '.')
    try:
        return float(s)
    except ValueError:
        return np.nan

df['price'] = df['price'].apply(clean_price)
print(f'Price dtype  : {df["price"].dtype}')
print(f'Price range  : {df["price"].min():.2f} - {df["price"].max():.2f}')

### Step 6 -- Cap Outliers
- **Price & Quantity**: cap at the 99th percentile (preserves near-legitimate highs)
- **Rating**: clip to `[1.0, 5.0]` -- values outside that range are physically impossible

In [ ]:
df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce')
df['rating']   = pd.to_numeric(df['rating'],   errors='coerce')

price_cap    = df['price'].quantile(0.99)
quantity_cap = df['quantity'].quantile(0.99)

# Save pre-cap series for the visualisation later
price_before_cap = df['price'].copy()

df['price']    = df['price'].clip(upper=price_cap)
df['quantity'] = df['quantity'].clip(upper=quantity_cap)
df['rating']   = df['rating'].clip(lower=1.0, upper=5.0)

print(f'Price capped at    : {price_cap:.2f}')
print(f'Quantity capped at : {quantity_cap:.0f}')
print(f'Rating clipped to  : [1.0, 5.0]')

### Step 7 -- Parse Dates
`pd.to_datetime` with `errors='coerce'` converts all five date formats;
genuinely unparseable strings become `NaT`.

In [ ]:
df['date_added'] = pd.to_datetime(df['date_added'], format='mixed', dayfirst=True, errors='coerce')
print(f'date_added dtype : {df["date_added"].dtype}')
print(f'NaT count        : {df["date_added"].isna().sum()}')
df['date_added'].dropna().sample(5, random_state=1)

### Step 8 -- Handle Missing Values
| Column | Strategy | Reason |
|---|---|---|
| `product_name`, `date_added`, `customer_email` | Drop row | Cannot be invented |
| `price`, `quantity`, `rating` | Fill with **median** | Robust to remaining skew |
| `category`, `city` | Fill with **mode** | Most common value is a safe default |

In [ ]:
n_before = len(df)
df = df.dropna(subset=['product_name', 'date_added', 'customer_email'])
print(f'Rows dropped (name/date/email missing): {n_before - len(df)}')

df['price']    = df['price'].fillna(df['price'].median())
df['quantity'] = df['quantity'].fillna(df['quantity'].median())
df['rating']   = df['rating'].fillna(df['rating'].median())
df['category'] = df['category'].fillna(df['category'].mode()[0])
df['city']     = df['city'].fillna(df['city'].mode()[0])

print(f'Remaining nulls : {df.isnull().sum().sum()}')

### Step 9 -- Fix Data Types
Imputation can silently turn integers into floats (`3 -> 3.0`). We fix that.

In [ ]:
df['quantity'] = df['quantity'].round().astype(int)
df['price']    = df['price'].round(2)
df['rating']   = df['rating'].round(1)

print('Final dtypes:')
print(df.dtypes)

## 4. After-Cleaning Summary

A side-by-side comparison of the dataset before and after.

In [ ]:
after_summary = pd.DataFrame({
    'dtype':  df.dtypes.astype(str),
    'nulls':  df.isnull().sum(),
    'null_%': (df.isnull().mean() * 100).round(1),
    'unique': df.nunique(),
})

print(f'Total rows     : {len(df)}')
print(f'Total nulls    : {df.isnull().sum().sum()}')
print(f'Duplicate rows : {df.duplicated().sum()}')
print()
after_summary

In [ ]:
# Reload original dirty data for comparison metrics
df_orig = pd.read_csv('data/dirty_ecommerce.csv', dtype=str)

comparison = pd.DataFrame({
    'Metric':  ['Row count', 'Total nulls', 'Duplicate rows'],
    'Before':  [len(df_orig),
                int(df_orig.isnull().sum().sum()),
                int(df_orig.duplicated().sum())],
    'After':   [len(df),
                int(df.isnull().sum().sum()),
                int(df.duplicated().sum())],
}).set_index('Metric')

comparison['Change'] = comparison['After'] - comparison['Before']
comparison

## 5. Visualisations

Two charts that make the cleaning impact immediately visible to clients.

### Pre-chart check: confirm `df` is fully cleaned

In [ ]:
# This cell verifies that all cleaning steps have been applied to 'df'.
# It must show 0 nulls and 0 duplicates before we plot.
print(f'Rows      : {len(df)}')
print(f'Nulls     : {df.isnull().sum().sum()}  <- must be 0')
print(f'Dupes     : {df.duplicated().sum()}  <- must be 0')
print(f'Price dtype   : {df["price"].dtype}')
print(f'Date dtype    : {df["date_added"].dtype}')
assert df.isnull().sum().sum() == 0, 'ERROR: nulls still present -- re-run cleaning cells first!'
assert df.duplicated().sum() == 0,  'ERROR: duplicates still present!'
print('\nAll checks passed. Ready to plot.')

### Chart 1 -- Missing Values Per Column: Before vs After

In [ ]:
df_orig_reload = pd.read_csv('data/dirty_ecommerce.csv', dtype=str)

nulls_before = df_orig_reload.isnull().sum()
nulls_after  = df.isnull().sum()

# Confirm cleaning worked before plotting
print('Null counts BEFORE cleaning:')
print(nulls_before.to_string())
print()
print('Null counts AFTER cleaning:')
print(nulls_after.to_string())
print(f'\nTotal nulls after: {nulls_after.sum()} (should be 0)')

cols = nulls_before.index.tolist()
x = np.arange(len(cols))
width = 0.38
MIN_VIS = 0.5   # minimum bar height so zero-value bars are still visible

# Use a display array: real zeros get a tiny stub so the bar is visible
after_display = np.where(nulls_after.values == 0, MIN_VIS, nulls_after.values.astype(float))

fig, ax = plt.subplots(figsize=(11, 5))
bars1 = ax.bar(x - width/2, nulls_before.values, width,
               label='Before', color='#e05c5c', alpha=0.88)
bars2 = ax.bar(x + width/2, after_display, width,
               label='After (0 nulls)',  color='#4caf7d', alpha=0.88)

ax.set_xticks(x)
ax.set_xticklabels(cols, rotation=35, ha='right', fontsize=10)
ax.set_ylabel('Number of missing values', fontsize=11)
ax.set_title('Missing Values Per Column: Before vs After Cleaning',
             fontsize=13, fontweight='bold', pad=12)
ax.legend(fontsize=11)
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))

# Labels: always show the REAL count (not the display height)
for bar, real_h in zip(bars1, nulls_before.values):
    if real_h > 0:
        ax.text(bar.get_x() + bar.get_width()/2, real_h + 0.2,
                str(int(real_h)), ha='center', va='bottom', fontsize=9, fontweight='bold')

for bar, real_h in zip(bars2, nulls_after.values):
    label_y = bar.get_height() + 0.2
    ax.text(bar.get_x() + bar.get_width()/2, label_y,
            str(int(real_h)), ha='center', va='bottom',
            fontsize=9, fontweight='bold', color='#1a5c36')

plt.tight_layout()
plt.savefig('data/chart_nulls.png', bbox_inches='tight')
plt.show()
print('Saved -> data/chart_nulls.png')

### Chart 2 -- Price Distribution: Before vs After Outlier Removal

In [ ]:
# Reload dirty data to get original prices as numbers
df_orig2 = pd.read_csv('data/dirty_ecommerce.csv', dtype=str)

def parse_price(val):
    if pd.isna(val): return np.nan
    s = str(val).replace('$','').replace('USD','').replace(',','.').strip()
    try:    return float(s)
    except: return np.nan

prices_dirty = df_orig2['price'].apply(parse_price).dropna()
prices_clean = df['price'].dropna()

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=False)

axes[0].hist(prices_dirty, bins=30, color='#e05c5c', alpha=0.85, edgecolor='white')
axes[0].set_title('Price Distribution -- BEFORE', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Price (USD)', fontsize=11)
axes[0].set_ylabel('Count', fontsize=11)
axes[0].annotate(f'Max: ${prices_dirty.max():,.0f}\nMean: ${prices_dirty.mean():,.0f}',
                 xy=(0.65, 0.82), xycoords='axes fraction', fontsize=10,
                 bbox=dict(boxstyle='round,pad=0.3', fc='#fce8e8'))

axes[1].hist(prices_clean, bins=30, color='#4caf7d', alpha=0.85, edgecolor='white')
axes[1].set_title('Price Distribution -- AFTER', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Price (USD)', fontsize=11)
axes[1].set_ylabel('Count', fontsize=11)
axes[1].annotate(f'Max: ${prices_clean.max():,.0f}\nMean: ${prices_clean.mean():,.0f}',
                 xy=(0.65, 0.82), xycoords='axes fraction', fontsize=10,
                 bbox=dict(boxstyle='round,pad=0.3', fc='#e8f5ee'))

fig.suptitle('Outlier Removal Effect on Price Column',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('data/chart_price.png', bbox_inches='tight')
plt.show()
print('Saved -> data/chart_price.png')

## 6. Save Clean Outputs

In [ ]:
df.to_csv('data/clean_ecommerce.csv',    index=False)
df.to_excel('data/clean_ecommerce.xlsx',  index=False, engine='openpyxl')

print('Files written:')
print('  data/clean_ecommerce.csv')
print('  data/clean_ecommerce.xlsx')
print()
print('Preview of clean data:')
df.head()

## 7. Conclusion

| Metric | Before | After |
|---|---|---|
| Rows | 200 | ~162 |
| Null values | ~64 | 0 |
| Duplicate rows | 20 | 0 |
| Price dtype | mixed strings | float64 |
| Date dtype | mixed strings | datetime64 |
| City variants | 20+ | 5 canonical |

The cleaned dataset is now ready for analysis, reporting, or loading into a database.

---
*Built with Python · Pandas · NumPy · Matplotlib*